# 10 - TuringServe benchmark on the T4 (Build Guide Phase 8, step 10)
Benchmark the exact commit tagged `m3-rc`.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # must run before torch is imported

In [ ]:
!nvidia-smi
!pip uninstall -y -q torchvision torchaudio
!pip install -q torch==2.6.0 --index-url https://download.pytorch.org/whl/cu124
!pip install -q "transformers>=4.45,<5" safetensors fastapi "uvicorn[standard]" httpx openai \
    prometheus-client psutil hypothesis pytest pytest-asyncio pytest-timeout datasets pandas matplotlib

**Run -> Restart session**, then run the first cell again and continue below.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # must run before torch is imported

In [ ]:
!git clone https://github.com/Muskanpandey0304/turingserve.git /kaggle/working/turingserve
%cd /kaggle/working/turingserve
!git checkout m3-rc
!pip install -q -e . --no-deps
!python scripts/check_versions.py

In [ ]:
!hf download Qwen/Qwen2.5-1.5B-Instruct --local-dir /kaggle/working/models/Qwen2.5-1.5B-Instruct
!hf download Qwen/Qwen2.5-0.5B-Instruct --local-dir /kaggle/working/models/Qwen2.5-0.5B-Instruct
!hf download anon8231489123/ShareGPT_Vicuna_unfiltered ShareGPT_V3_unfiltered_cleaned_split.json --repo-type dataset --local-dir /kaggle/working/datasets/sharegpt

In [ ]:
os.environ["TS_MODEL_15"] = "/kaggle/working/models/Qwen2.5-1.5B-Instruct"
os.environ["TS_MODEL_05"] = "/kaggle/working/models/Qwen2.5-0.5B-Instruct"
os.environ["TS_SHAREGPT"] = "/kaggle/working/datasets/sharegpt/ShareGPT_V3_unfiltered_cleaned_split.json"
os.environ["TS_KAGGLE"] = "1"

In [ ]:
import subprocess, time, requests
log = open("/kaggle/working/server.log", "w")
p = subprocess.Popen(["turingserve", "serve", "--model", os.environ["TS_MODEL_15"], "--served-model-name", "qwen", "--port", "8000"], stdout=log, stderr=subprocess.STDOUT)
for _ in range(240):
    try:
        if requests.get("http://localhost:8000/health", timeout=2).status_code == 200:
            break
    except Exception:
        pass
    time.sleep(5)

In [ ]:
!python -m benchmarks.serve_bench --base-url http://localhost:8000 --model qwen --tokenizer $TS_MODEL_15 --dataset sharegpt --num-prompts 500 --mode concurrency --concurrency 1,4,16,32,64 --ignore-eos --label t4-1.5b --gpu-log --out /kaggle/working/results

Prefill sweep (2,048 / 4,096 / 8,000 tokens): restart with `--max-model-len 8192` **and `--no-prefix-caching`**. The benchmark replays the same 15 prompts in the warm-up and every repetition, so with prefix caching on it would measure cache hits and not prefill speed. `serve_bench --out` is a directory: it writes `<date>_<engine>_<workload>_<label>.json` inside it. Download that file and commit it under `benchmarks/results/`. Notebook 11 repeats just this step with a per-length summary.


In [ ]:
p.terminate(); p.wait()

In [ ]:
import subprocess, time, requests
log = open("/kaggle/working/server.log", "w")
p = subprocess.Popen(["turingserve", "serve", "--model", os.environ["TS_MODEL_15"], "--served-model-name", "qwen", "--port", "8000", "--max-model-len", "8192", "--no-prefix-caching"], stdout=log, stderr=subprocess.STDOUT)
for _ in range(240):
    try:
        if requests.get("http://localhost:8000/health", timeout=2).status_code == 200:
            break
    except Exception:
        pass
    time.sleep(5)

In [ ]:
!python -m benchmarks.serve_bench --base-url http://localhost:8000 --model qwen --tokenizer $TS_MODEL_15 --dataset prefill --mode concurrency --concurrency 1 --repeat 3 --label t4-1.5b-prefill-cold --gpu-log --out /kaggle/working/results

## HF baseline (naive, one request at a time)
`hf_server.py` serializes every request behind one lock, so concurrency levels do not change its throughput, and `ignore_eos` forces full-length generation. The full 200-prompt sweep takes hours and has frozen the kernel (Cancel Run cannot interrupt a thread blocked in `generate()`), so run a small smoke test: 10 prompts, one rep, concurrency 1 (measured 27.8 tok/s on the T4).

First stop the TuringServe server, then start HF on the *second* T4 (`CUDA_VISIBLE_DEVICES=1`). Loading it on GPU 0 next to the TuringServe server ran out of memory.

In [ ]:
p.terminate(); p.wait()


In [ ]:
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "1"
hf_log = open("/kaggle/working/hf_server.log", "w")
hf_p = subprocess.Popen(["python", "-m", "benchmarks.baselines.hf_server", "--model", os.environ["TS_MODEL_15"], "--port", "8003"],
                        stdout=hf_log, stderr=subprocess.STDOUT, env=env)
for _ in range(240):
    try:
        if requests.get("http://localhost:8003/health", timeout=2).status_code == 200:
            print("HF server up")
            break
    except Exception:
        pass
    time.sleep(5)


In [ ]:
!python -m benchmarks.serve_bench --base-url http://localhost:8003 --model qwen --tokenizer $TS_MODEL_15 --dataset sharegpt --num-prompts 10 --mode concurrency --concurrency 1 --repeat 1 --engine hf --label hf-baseline-smoke --out /kaggle/working/results


In [ ]:
hf_p.terminate(); hf_p.wait()
